# 임베딩 모델 비교 — multilingual-e5-large vs KoE5 (+ KURE)

**목적**: PR #19 (`findings_2026-06-02.md`) 의 결정적 발견 — multilingual-e5-large 가 한국어 의미보다 글자 패턴에 cluster 됨 — 의 본질 해결 검증. 같은 60k 단어에 한국어 특화 모델 (KoE5) 임베딩 새로 만들고 동일 spot-check.

**핵심 검증 질문**:
1. KoE5 에서도 "사과" top 50 의 다수가 "-과" 끝 단어인가?
2. KoE5 에서 사과↔배 cosine 이 사과↔자동차 보다 높아지나?
3. KoE5 에서 과일 cross-check rank 가 개선되나?
4. KoE5 에서 "강아지·학교·사랑" 의 정상 cluster 가 유지되나? (회귀 없음 확인)

**전제**: `data/embedding_dictionary_koe5.json` 이미 생성됨. 안 됐으면 먼저:
```
python tools/embedding_eval/build_alt_embeddings.py \
  --model nlpai-lab/KoE5 \
  --output data/embedding_dictionary_koe5.json
```

## 1. Setup — 두 임베딩 사전 로드

In [7]:
import pltfont
pltfont.auto()

✅ 현재 폰트 적용됨: Arial Unicode MS


In [8]:
import json
from pathlib import Path
import numpy as np

REPO_ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents)
                 if (p / 'data' / 'embedding_dictionary_e5.json').exists())

def load_emb(path):
    """임베딩 JSON 로드 → (words, matrix (L2 normalized), word_to_idx)."""
    with open(path, 'rb') as f:
        d = json.loads(f.read())
    words = list(d.keys())
    mat = np.array([d[w] for w in words], dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    mat = mat / norms
    return words, mat, {w: i for i, w in enumerate(words)}

print('e5 (현행) 로드...')
e5_words, e5_mat, e5_idx = load_emb(REPO_ROOT / 'data' / 'embedding_dictionary_e5.json')
print(f'  e5: {len(e5_words):,} 단어, shape={e5_mat.shape}')

koe5_path = REPO_ROOT / 'data' / 'embedding_dictionary_koe5.json'
if not koe5_path.exists():
    print(f'\n❌ KoE5 임베딩 없음. 먼저:')
    print(f'   python tools/embedding_eval/build_alt_embeddings.py \\')
    print(f'     --model nlpai-lab/KoE5 \\')
    print(f'     --output data/embedding_dictionary_koe5.json')
else:
    print('\nKoE5 로드...')
    koe5_words, koe5_mat, koe5_idx = load_emb(koe5_path)
    print(f'  KoE5: {len(koe5_words):,} 단어, shape={koe5_mat.shape}')

# (선택) KURE 도 있으면 로드
kure_path = REPO_ROOT / 'data' / 'embedding_dictionary_kure.json'
if kure_path.exists():
    print('\nKURE 로드...')
    kure_words, kure_mat, kure_idx = load_emb(kure_path)
    print(f'  KURE: {len(kure_words):,} 단어, shape={kure_mat.shape}')
else:
    kure_words = kure_mat = kure_idx = None
    print('\nKURE 없음 (선택사항)')

e5 (현행) 로드...
  e5: 60,000 단어, shape=(60000, 1024)

KoE5 로드...
  KoE5: 60,000 단어, shape=(60000, 1024)

KURE 로드...
  KURE: 60,000 단어, shape=(60000, 1024)


## 2. Helper — 모델별 top N · pair cosine · 카테고리 cross-check

In [9]:
def top_n(words, mat, idx, answer, n=20, exclude_self=True):
    if answer not in idx:
        return None  # 사전에 없으면 skip
    vec = mat[idx[answer]]
    sims = mat @ vec
    order = np.argsort(-sims)
    skip = 1 if exclude_self else 0
    return [(words[i], float(sims[i])) for i in order[skip:skip + n]]

def pair_cos(mat, idx, a, b):
    if a not in idx or b not in idx:
        return float('nan')
    return float(mat[idx[a]] @ mat[idx[b]])

def rank_of(words, mat, idx, answer, candidates):
    """answer 의 sim 순위에서 candidates 각각의 rank + sim."""
    if answer not in idx:
        return None
    vec = mat[idx[answer]]
    sims = mat @ vec
    order = np.argsort(-sims)
    rank_map = {words[i]: r for r, i in enumerate(order, 1)}
    out = []
    for c in candidates:
        if c not in idx:
            out.append((c, -1, float('nan')))
        else:
            out.append((c, rank_map[c], float(sims[idx[c]])))
    return out

def print_top_side_by_side(answer, n=20):
    models = [('e5 (multilingual-e5)', e5_words, e5_mat, e5_idx)]
    if 'koe5_mat' in globals() and koe5_mat is not None:
        models.append(('KoE5', koe5_words, koe5_mat, koe5_idx))
    if kure_mat is not None:
        models.append(('KURE', kure_words, kure_mat, kure_idx))
    
    for label, w, m, i in models:
        print(f'\n--- "{answer}" top {n} — {label} ---')
        res = top_n(w, m, i, answer, n)
        if res is None:
            print('  (사전에 없음)')
            continue
        for r, (word, sim) in enumerate(res, 1):
            print(f'  {r:>2}. {word:<12} {sim:.4f}')

## 3. 결정적 fail case 재시험 — 사과↔배 vs 사과↔자동차

**PR #19 발견**: e5 에서 사과↔배=0.1239, 사과↔자동차=0.1668 (자동차가 더 가까움!).
KoE5 에서 정상화되나?

In [10]:
pairs = [('사과', '배'), ('사과', '자동차'), ('사과', '포도'), ('사과', '딸기'),
         ('강아지', '고양이'), ('강아지', '자동차')]

header = f'{"pair":<20} {"e5":>10}'
if 'koe5_mat' in globals() and koe5_mat is not None:
    header += f' {"KoE5":>10}'
if kure_mat is not None:
    header += f' {"KURE":>10}'
print(header)
print('-' * len(header))

for a, b in pairs:
    e5_c = pair_cos(e5_mat, e5_idx, a, b)
    line = f'{a + "↔" + b:<20} {e5_c:>10.4f}'
    if 'koe5_mat' in globals() and koe5_mat is not None:
        c = pair_cos(koe5_mat, koe5_idx, a, b)
        line += f' {c:>10.4f}'
    if kure_mat is not None:
        c = pair_cos(kure_mat, kure_idx, a, b)
        line += f' {c:>10.4f}'
    print(line)

print('\n해석 가이드:')
print('- 사과↔배 > 사과↔자동차 가 정상')
print('- 사과↔포도/딸기 도 사과↔자동차 보다 커야 정상')
print('- 강아지↔고양이 > 강아지↔자동차 가 정상')
print('  (cosine 절대치는 모델 간 비교 X, relative 만 의미 있음)')

pair                         e5       KoE5       KURE
-----------------------------------------------------
사과↔배                     0.1239     0.6450     0.4670
사과↔자동차                   0.1668     0.6609     0.4656
사과↔포도                    0.2234     0.6678     0.4886
사과↔딸기                    0.1556     0.6778     0.4483
강아지↔고양이                  0.4946     0.7816     0.7902
강아지↔자동차                  0.2696     0.6531     0.5711

해석 가이드:
- 사과↔배 > 사과↔자동차 가 정상
- 사과↔포도/딸기 도 사과↔자동차 보다 커야 정상
- 강아지↔고양이 > 강아지↔자동차 가 정상
  (cosine 절대치는 모델 간 비교 X, relative 만 의미 있음)


## 4. top N 비교 — 사과·배·강아지·학교·사랑

**관찰 포인트**:
- e5 의 "-과 끝" cluster (제과·인과·안과 등) 가 KoE5 에서 사라지나?
- e5 의 "배-" 시작 cluster (배장·배압·배부 등) 가 KoE5 에서 사라지나?
- e5 가 잘했던 "강아지"·"학교"·"사랑" 의 의미 cluster 가 KoE5 에서도 유지되나? (회귀 X)

In [11]:
for ans in ['사과', '배', '강아지', '학교', '사랑', '집']:
    print_top_side_by_side(ans, n=20)
    print()


--- "사과" top 20 — e5 (multilingual-e5) ---
   1. 사과와          0.7390
   2. 사과는          0.6446
   3. 사과가          0.6373
   4. 사과의          0.5365
   5. 설과           0.5316
   6. 사과를          0.5247
   7. 제과           0.5071
   8. 인과           0.5049
   9. 사이다          0.5040
  10. 여과           0.4870
  11. 무과           0.4508
  12. 사바           0.4441
  13. 신과           0.4344
  14. 사사           0.4294
  15. 안과           0.4195
  16. 내과           0.4179
  17. 사실과          0.4161
  18. 일과           0.4031
  19. 사람과          0.4024
  20. 정과           0.3969

--- "사과" top 20 — KoE5 ---
   1. 사과는          0.9613
   2. 사과를          0.9447
   3. 사과와          0.9373
   4. 사과가          0.8551
   5. 대과           0.8400
   6. 사이다          0.8185
   7. 사과나무         0.8141
   8. 무과           0.7993
   9. 제과           0.7931
  10. 사과문을         0.7824
  11. 사과의          0.7811
  12. 과일           0.7783
  13. 과일과          0.7752
  14. 설과           0.7747
  15. 과이다          0.7724
  16. 오과다         

### 발견 — top N 비교 (사용자가 채움)

- "사과" e5: ___ → KoE5: ___
- "배" e5: ___ → KoE5: ___
- "강아지" 회귀 여부: ___
- "학교"·"사랑" 회귀 여부: ___

## 5. 과일 카테고리 cross-check — 모델 비교

**PR #19 발견**: e5 에서 사과 정답에서 배 3,888위, 포도 737위 등 도메인 cluster fail.
KoE5 에서 과일들이 top 100/500 안에 들어오나?

In [12]:
ANSWER = '사과'
FRUITS = ['배', '포도', '바나나', '딸기', '수박', '망고', '레몬', '복숭아', '귤', '참외']

e5_ranks = rank_of(e5_words, e5_mat, e5_idx, ANSWER, FRUITS)
koe5_ranks = rank_of(koe5_words, koe5_mat, koe5_idx, ANSWER, FRUITS) if 'koe5_mat' in globals() and koe5_mat is not None else None
kure_ranks = rank_of(kure_words, kure_mat, kure_idx, ANSWER, FRUITS) if kure_mat is not None else None

header = f'{"fruit":<10} {"e5 rank":>10} {"e5 cos":>10}'
if koe5_ranks: header += f' {"KoE5 rank":>12} {"KoE5 cos":>10}'
if kure_ranks: header += f' {"KURE rank":>12} {"KURE cos":>10}'
print(f'정답: "{ANSWER}" — 다른 과일들의 순위 + cosine\n')
print(header)
print('-' * len(header))

for i, (fruit, r, s) in enumerate(e5_ranks):
    r_str = '없음' if r < 0 else f'{r:>10,}'
    s_str = '-' if s != s else f'{s:>10.4f}'
    line = f'{fruit:<10} {r_str:>10} {s_str}'
    if koe5_ranks:
        _, kr, ks = koe5_ranks[i]
        kr_str = '없음' if kr < 0 else f'{kr:>12,}'
        ks_str = '-' if ks != ks else f'{ks:>10.4f}'
        line += f' {kr_str} {ks_str}'
    if kure_ranks:
        _, ur, us = kure_ranks[i]
        ur_str = '없음' if ur < 0 else f'{ur:>12,}'
        us_str = '-' if us != us else f'{us:>10.4f}'
        line += f' {ur_str} {us_str}'
    print(line)

정답: "사과" — 다른 과일들의 순위 + cosine

fruit         e5 rank     e5 cos    KoE5 rank   KoE5 cos    KURE rank   KURE cos
--------------------------------------------------------------------------------
배               3,888     0.1239        2,590     0.6450       17,708     0.4670
포도                737     0.2234          874     0.6678        9,732     0.4886
바나나               125     0.3092          118     0.7110       15,001     0.4737
딸기              2,316     0.1556          563     0.6778       26,179     0.4483
수박                992     0.2057          244     0.6965       27,015     0.4465
망고              1,859     0.1687          472     0.6819       24,150     0.4526
레몬              1,574     0.1793          453     0.6829       25,016     0.4507
복숭아             1,270     0.1915        1,099     0.6630       38,766     0.4198
귤                  없음 - 없음 - 없음 -
참외                 없음 - 없음 - 없음 -


### 발견 — 과일 cross-check (사용자가 채움)

- 과일 중 top 100 안에 들어온 개수 (e5): ___ → KoE5: ___
- 평균 rank 개선: ___
- 가장 큰 개선: ___

## 6. 종합 결정 (사용자가 채움)

### 6.1 한 줄 결론
- KoE5 가 의미 cluster 개선했나? : ___
- multilingual-e5-large 의 글자 패턴 cluster 해소? : ___
- 정상 cluster (강아지·학교) 회귀 여부? : ___

### 6.2 모델 교체 결정
- [ ] **교체** — KoE5 가 본질적으로 더 나아 보임. 다음 PR `feat/swap-to-koe5` (사전 교체 + game 재테스트 + HF 재업로드)
- [ ] **부분 교체** — KoE5 도 일부 개선만. 다른 옵션 (KURE 추가 비교, scaling step 검증)
- [ ] **교체 X** — KoE5 도 비슷. 임베딩 모델보다 다른 영역 (sim_calibration·prompt) 우선

### 6.3 새 발견
- (위에 없는 패턴) ___